# SecurityScorecard Hackathon - Product Usage Search Tool
This tool helps track down vendors or suppliers that might be affected by a product-related breach, vulnerability, or outage.  
Examples: CrowdStrike Falcon, SolarWinds, Kaseya VSA, Log4j

In [1]:
# Reset - Delete contents of output and products folders using Linux commands
!rm -rf output/* products/*
print("Successfully deleted contents of 'output' and 'products' directories")

Successfully deleted contents of 'output' and 'products' directories


In [2]:
# Data Collection from SSC APIs - Domain response and product data

# Import libraries
import csv
import requests
import os
import json
import time
import re
import glob
from datetime import datetime
import concurrent.futures
from tqdm import tqdm
import pandas as pd
from ipywidgets import widgets
from IPython.display import display, HTML, clear_output

# Create output directories if they don't exist
os.makedirs('output', exist_ok=True)
os.makedirs('products', exist_ok=True)

# API configuration from environment variables
import os
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv()

# Get API configuration from environment variables
API_TOKEN = os.getenv("SSC_API_TOKEN")
PORTFOLIO_ID = os.getenv("SSC_PORTFOLIO_ID")

# Validate that required environment variables are set
if not API_TOKEN:
    raise ValueError("SSC_API_TOKEN environment variable is not set. Please check your .env file.")
if not PORTFOLIO_ID:
    raise ValueError("SSC_PORTFOLIO_ID environment variable is not set. Please check your .env file.")

print(f"API credentials loaded successfully. Using portfolio: {PORTFOLIO_ID}")

def add_company_to_portfolio(domain):
    """Add a company to the portfolio and return the response"""
    url = f"https://api.securityscorecard.io/portfolios/{PORTFOLIO_ID}/companies/{domain}"
    
    headers = {
        "accept": "application/json; charset=utf-8",
        "Authorization": f"Token {API_TOKEN}"
    }
    
    response = requests.put(url, headers=headers)
    
    try:
        return response.json()
    except json.JSONDecodeError:
        print(f"Error decoding JSON for domain {domain}: {response.text}")
        return {"error": response.text}

def get_domain_products(domain):
    """Get products for a domain"""
    url = f"https://api.securityscorecard.io/vendor-detection/{domain}/products"
    
    headers = {
        "accept": "application/json",
        "Authorization": f"Token {API_TOKEN}"
    }
    
    response = requests.get(url, headers=headers)
    
    try:
        return response.json()
    except json.JSONDecodeError:
        print(f"Error decoding JSON for domain {domain}: {response.text}")
        return {"error": response.text}

def domain_to_filename(domain):
    """Convert domain to a filename-friendly format"""
    return domain.replace('.', '_')

def process_single_domain(domain, timestamp):
    """Process a single domain and return the result"""
    # Add company to portfolio
    response = add_company_to_portfolio(domain)
    
    # Extract relevant data from response
    result = {
        'domain_input': domain,
        'domain_response': response.get('domain', ''),
        'name_response': response.get('name', ''),
        'score_response': response.get('score', ''),
        'grade_response': response.get('grade', ''),
        'industry_response': response.get('industry', ''),
        'size_response': response.get('size', ''),
        'timestamp': datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    }
    
    # Process products if we have a response domain
    response_domain = result['domain_response']
    if response_domain:
        # Get products for the domain
        products = get_domain_products(response_domain)
        
        # Create filename based on domains
        input_domain = domain
        if input_domain.lower() == response_domain.lower():
            filename = f"{domain_to_filename(input_domain)}_{timestamp}.csv"
        else:
            filename = f"{domain_to_filename(input_domain)}_OR_{domain_to_filename(response_domain)}_{timestamp}.csv"
        
        # Save products to CSV
        products_path = f"products/{filename}"
        
        # Extract product names and save as a simple CSV
        with open(products_path, 'w', newline='', encoding='utf-8') as products_file:
            writer = csv.writer(products_file)
            writer.writerow(['product_list'])
            
            # Handle the case where products is a dict with 'entries' key
            if isinstance(products, dict) and 'entries' in products:
                for entry in products['entries']:
                    if isinstance(entry, dict) and 'name' in entry:
                        writer.writerow([entry['name']])
            # Handle other cases
            else:
                writer.writerow(["Raw API response (unexpected format)"])
                writer.writerow([str(products)])
    
    return result

def process_domains(input_csv_path, max_workers=10):
    """Process domains from input CSV and save results using concurrency"""
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    output_csv_path = f"output/domain_check_results_{timestamp}.csv"
    
    # Read input domains
    with open(input_csv_path, 'r') as input_file:
        reader = csv.reader(input_file)
        domains = [row[0] for row in reader if row]  # Assuming first column contains domains
    
    total_domains = len(domains)
    print(f"Processing {total_domains} domains...")
    
    results = []
    
    # Process domains with progress bar
    with concurrent.futures.ThreadPoolExecutor(max_workers=max_workers) as executor:
        # Submit all tasks
        future_to_domain = {executor.submit(process_single_domain, domain, timestamp): domain for domain in domains}
        
        # Process results as they complete with tqdm progress bar
        for future in tqdm(concurrent.futures.as_completed(future_to_domain), total=len(domains), desc="Processing domains"):
            domain = future_to_domain[future]
            try:
                result = future.result()
                results.append(result)
            except Exception as exc:
                print(f"Error processing {domain}: {exc}")
                with open("error_log.txt", "a", encoding='utf-8') as error_file:
                    error_file.write(f"{domain}: {exc}\n")
    
    # Save results to CSV
    if results:
        with open(output_csv_path, 'w', newline='', encoding='utf-8') as output_file:
            fieldnames = results[0].keys()
            writer = csv.DictWriter(output_file, fieldnames=fieldnames)
            writer.writeheader()
            writer.writerows(results)
        
        print(f"Domain check results saved to {output_csv_path}")
        
        # Display a sample of results
        sample_size = min(5, len(results))
        print(f"\nSample of {sample_size} results:")
        display(HTML(pd.DataFrame(results[:sample_size]).to_html()))
    else:
        print("No results to save.")
    
    return output_csv_path

API credentials loaded successfully. Using portfolio: 458ecb3b-7c38-5312-8cd0-a797b3e3e4ca


In [3]:
process_domains("input_domains.csv") # Can be a vendor list or domains taken from a portfolio

Processing 30 domains...


Processing domains: 100%|██████████| 30/30 [00:05<00:00,  5.03it/s]

Domain check results saved to output/domain_check_results_20250503_013021.csv

Sample of 5 results:


,domain_input,domain_response,name_response,score_response,grade_response,industry_response,size_response,timestamp
0,sap.com,sap.com,SAP,85,B,technology,size_more_than_10000,2025-05-03 01:30:23
1,hp.com,hp.com,HP,96,A,technology,size_more_than_10000,2025-05-03 01:30:22
2,microsoft.com,microsoft.com,Microsoft Corporation,73,C,technology,size_more_than_10000,2025-05-03 01:30:23
3,cisco.com,cisco.com,Cisco Systems Inc,81,B,technology,size_more_than_10000,2025-05-03 01:30:22
4,amazon.com,amazon.com,Amazon,79,C,retail,size_more_than_10000,2025-05-03 01:30:23


'output/domain_check_results_20250503_013021.csv'

In [4]:
# Optimization - Cache all product data to make searches faster
def load_all_product_data():
    """Load all product data into memory for faster searching"""
    all_data = []
    product_files = glob.glob('products/*.csv')
    
    for file_path in product_files:
        domains = get_domain_from_filename(file_path)
        
        try:
            with open(file_path, 'r', newline='') as csvfile:
                reader = csv.reader(csvfile)
                next(reader)  # Skip header row
                
                for row in reader:
                    if not row or not row[0].strip():
                        continue
                        
                    product = row[0]
                    
                    for domain in domains:
                        all_data.append({
                            'domain': domain,
                            'product': product,
                            'file': file_path
                        })
        except Exception as e:
            print(f"Error processing {file_path}: {e}")
    
    return pd.DataFrame(all_data)

def get_domain_from_filename(filename):
    """Extract domain name from product CSV filename"""
    # Extract just the filename without path and extension
    base_name = os.path.basename(filename).replace('.csv', '')
    
    # Remove timestamp portion (assuming format like domain_20250502_052439.csv)
    domain_part = re.sub(r'_\d{8}_\d{6}$', '', base_name)
    
    # Replace underscores with dots for normal domains, but handle OR case specially
    if '_OR_' in domain_part:
        parts = domain_part.split('_OR_')
        return [part.replace('_', '.') for part in parts]
    else:
        return [domain_part.replace('_', '.')]

def search_cached_data(search_term, case_sensitive=False, df=None):
    """Search the cached product data for a specific term"""
    if df is None:
        return pd.DataFrame()
        
    if not search_term.strip():
        return pd.DataFrame()
    
    if case_sensitive:
        return df[df['product'].str.contains(search_term)]
    else:
        return df[df['product'].str.lower().str.contains(search_term.lower())]

In [5]:
# Load all product data into memory
print("Loading product data...")
all_product_data = load_all_product_data()
print(f"Loaded {len(all_product_data)} product entries from {all_product_data['file'].nunique()} files")
print(f"Found {all_product_data['domain'].nunique()} unique domains")
print("Ready for Real Time Search & Data Export!")

Loading product data...
Loaded 42456 product entries from 29 files
Found 29 unique domains
Ready for Real Time Search & Data Export!


# Real Time Search, Visualizations and Data Export

In [6]:
# Search - Create output widget for displaying results
output_widget = widgets.Output()
stats_output = widgets.Output()

# Import visualization libraries
import matplotlib.pyplot as plt

# Create tab layout for different views
tab = widgets.Tab([output_widget, stats_output])
tab.set_title(0, 'Search Results')
tab.set_title(1, 'Usage Statistics')

# Create search widget
search_input = widgets.Text(description="Search:", placeholder="Type to search products...")
case_sensitive_checkbox = widgets.Checkbox(description="Case sensitive", value=False)
export_button = widgets.Button(description="Export Results", disabled=True)
status_label = widgets.Label(value="Type to search...")

# Function to update results as user types
def on_search_change(change):
    with output_widget:
        clear_output(wait=True)
        search_term = change['new']
        case_sensitive = case_sensitive_checkbox.value
        
        if not search_term.strip():
            status_label.value = "Type to search..."
            export_button.disabled = True
            return
        
        results = search_cached_data(search_term, case_sensitive, all_product_data)
        
        if len(results) > 0:
            status_label.value = f"Found {len(results)} matches across {results['domain'].nunique()} domains"
            export_button.disabled = False
            
            # Create a consolidated view with domain and product count
            domain_counts = results.groupby('domain').size().reset_index(name='product_count')
            
            # Merge the counts with the detailed results
            consolidated_results = results.merge(domain_counts, on='domain')
            
            # Sort by product count (descending) and then by domain
            consolidated_results = consolidated_results.sort_values(['product_count', 'domain'], ascending=[False, True])
            
            # Display single consolidated table
            print(f"Search results for '{search_term}':")
            display(consolidated_results[['domain', 'product_count', 'product', 'file']].head(100))
            
            if len(consolidated_results) > 100:
                print(f"\n... and {len(consolidated_results) - 100} more matches (export to see all)")
            
            # Update statistics
            update_usage_statistics(results, search_term)
        else:
            status_label.value = "No matches found"
            export_button.disabled = True
            print("No matches found for your search term.")
            
            # Clear statistics
            with stats_output:
                clear_output(wait=True)
                print("No data to analyze.")

# Function to create simple usage statistics
def update_usage_statistics(results, search_term):
    with stats_output:
        clear_output(wait=True)
        
        # Get unique domains and their product counts
        domain_counts = results.groupby('domain').size().reset_index(name='product_count')
        domain_counts = domain_counts.sort_values('product_count', ascending=False)
        
        # Calculate key metrics
        total_suppliers = all_product_data['domain'].nunique()
        matching_suppliers = results['domain'].nunique()
        percentage = (matching_suppliers / total_suppliers * 100) if total_suppliers > 0 else 0
        
        # Display key statistics
        print(f"\n--- Usage Statistics for '{search_term}' ---\n")
        print(f"• {matching_suppliers} out of {total_suppliers} suppliers use this product ({percentage:.1f}%)")
        
        if len(domain_counts) > 0:
            # Top suppliers
            print(f"\n--- Top Suppliers Using '{search_term}' ---")
            for i, (_, row) in enumerate(domain_counts.head(10).iterrows()):
                print(f"{i+1}. {row['domain']} ({row['product_count']} instances)")
            
            # Create simple bar chart of top suppliers
            plt.figure(figsize=(10, 6))
            
            # Plot top 15 suppliers or all if less than 15
            plot_data = domain_counts.head(min(15, len(domain_counts)))
            bars = plt.bar(plot_data['domain'], plot_data['product_count'], color='skyblue')
            
            # Add count labels on top of bars
            for bar in bars:
                height = bar.get_height()
                plt.text(bar.get_x() + bar.get_width()/2., height + 0.1,
                        f'{int(height)}', ha='center', va='bottom')
            
            plt.title(f'Top Suppliers Using "{search_term}"', fontsize=14)
            plt.xlabel('Supplier Domain')
            plt.ylabel('Number of Instances')
            plt.xticks(rotation=45, ha='right')
            plt.tight_layout()
            plt.show()
            
            # Create a simple pie chart showing percentage of suppliers
            plt.figure(figsize=(8, 8))
            labels = ['Using Product', 'Not Using Product']
            sizes = [matching_suppliers, total_suppliers - matching_suppliers]
            colors = ['#5DA5DA', '#F15854'] 
            explode = (0.1, 0)  # explode the 1st slice
            
            plt.pie(sizes, explode=explode, labels=labels, colors=colors,
                   autopct='%1.1f%%', shadow=True, startangle=90)
            plt.axis('equal')  # Equal aspect ratio ensures that pie is drawn as a circle
            plt.title(f'Supplier Adoption of "{search_term}"', fontsize=14)
            plt.show()
            
            # Display summary HTML with key metrics
            summary_html = f"""
            <div style="background-color:#f8f9fa; padding:15px; border-radius:5px; margin-top:15px;">
                <h3>Summary for "{search_term}"</h3>
                <div style="display:flex; justify-content:space-between;">
                    <div style="flex:1; padding:10px; background-color:#e9f5e9; border-radius:5px; margin-right:5px; display:flex; flex-direction:column; justify-content:center; align-items:center;">
                        <h1 style="margin:0; color:#2E8B57;">{matching_suppliers}</h1>
                        <p style="text-align:center; width:100%;">Suppliers using this product</p>
                    </div>
                    
                    <div style="flex:1; padding:10px; background-color:#e9f5e9; border-radius:5px; margin-right:5px; display:flex; flex-direction:column; justify-content:center; align-items:center;">
                        <h1 style="margin:0; color:#2E8B57;">{percentage:.1f}%</h1>
                        <p style="text-align:center; width:100%;">Of your supplier base</p>
                    </div>
                    
                    <div style="flex:1; padding:10px; background-color:#e9f5e9; border-radius:5px; display:flex; flex-direction:column; justify-content:center; align-items:center;">
                        <h1 style="margin:0; color:#2E8B57;">{domain_counts.iloc[0]['domain'] if len(domain_counts) > 0 else 'N/A'}</h1>
                        <p style="text-align:center; width:100%;">Top supplier by usage</p>
                    </div>
                </div>
            </div>
            """
            display(HTML(summary_html))

# Function to handle case sensitivity changes
def on_case_change(change):
    # Trigger search update when case sensitivity changes
    on_search_change({'new': search_input.value})

# Function to export results
def on_export_click(b):
    search_term = search_input.value
    case_sensitive = case_sensitive_checkbox.value
    results = search_cached_data(search_term, case_sensitive, all_product_data)
    
    if len(results) > 0:
        filename = f"{search_term.replace(' ', '_')}_results.csv"
        results.to_csv(filename, index=False)
        status_label.value = f"Exported {len(results)} results to {filename}"
    else:
        status_label.value = "No results to export"

# Connect event handlers
search_input.observe(on_search_change, names='value')  # This makes it update as you type
case_sensitive_checkbox.observe(on_case_change, names='value')
export_button.on_click(on_export_click)

# Display widgets
display(widgets.HBox([search_input, case_sensitive_checkbox, export_button]))
display(status_label)
display(tab)  # Display tabs instead of just output_widget

Label(value='Type to search...')